# BOJ_3 Custom Feature Analysis

以下の特徴量設計で分析を行います：
1. BOJスワップ: TONAスプレッド化し、前日値 + 2-5日MA乖離を使用。
2. その他項目: 前日値は含めず、2-5日MA乖離のみを使用。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. データの読み込み
df = pd.read_excel('data/BOJ_data.xlsx')
df = df.iloc[1:].copy()
df['日付'] = pd.to_datetime(df['日付'], format='%Y年%m月%d日')

swap_cols = [f'JPBOJ{i}ONI=TRDT (MID_PRICE)' for i in range(1, 9)]
tona_col = 'JPY1DOIS=ICAP (MID_PRICE)'
jpy_col = 'JPY= (MID_PRICE)'
market_cols = ['JGBc1 (TRDPRC_1)', '.N225 (TRDPRC_1)', '.DXY (TRDPRC_1)']

all_cols = swap_cols + [tona_col, jpy_col] + market_cols
for col in all_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values('日付').reset_index(drop=True)
df.dropna(subset=[swap_cols[0], tona_col], inplace=True)
print(f'Loaded {len(df)} rows')

In [ ]:
# 2. MPM日程
mpm_dates = pd.to_datetime([
    '2024-01-23', '2024-03-19', '2024-04-26', '2024-06-14', '2024-07-31', '2024-09-20', '2024-10-31', '2024-12-19',
    '2025-01-24', '2025-03-19', '2025-04-30', '2025-06-17', '2025-07-31', '2025-09-19', '2025-10-30', '2025-12-19',
    '2026-01-23', '2026-03-19', '2026-04-28', '2026-06-16', '2026-07-31', '2026-09-18', '2026-10-30', '2026-12-18'
])
def get_next_mpm(d):
    future = mpm_dates[mpm_dates > d]
    return future[0] if len(future) > 0 else None
df['Next_MPM'] = df['日付'].apply(get_next_mpm)
df['DaysToNextMPM'] = (df['Next_MPM'] - df['日付']).dt.days

In [ ]:
# 3. カスタム特徴量生成
features = []
df_feats = df.copy()

# A. BOJスワップのTONAスプレッド化と特徴量
for i, col in enumerate(swap_cols):
    spread_col = f'BOJ{i+1}_Spread'
    df_feats[spread_col] = df_feats[col] - df_feats[tona_col]
    
    # 前日値 (lag1)
    df_feats[f'{spread_col}_lag1'] = df_feats[spread_col].shift(1)
    features.append(f'{spread_col}_lag1')
    
    # 2-5日MA乖離 (lag1 - MA_n)
    for w in [2, 3, 4, 5]:
        ma = df_feats[spread_col].shift(1).rolling(window=w).mean()
        feat_name = f'{spread_col}_diff_MA{w}'
        df_feats[feat_name] = df_feats[f'{spread_col}_lag1'] - ma
        features.append(feat_name)

# B. その他項目 (前日値は入れず、2-5日MA乖離のみ)
other_items = [jpy_col] + market_cols
for col in other_items:
    lag1 = df_feats[col].shift(1)
    for w in [2, 3, 4, 5]:
        ma = df_feats[col].shift(1).rolling(window=w).mean()
        feat_name = f'{col}_diff_MA{w}'
        df_feats[feat_name] = lag1 - ma
        features.append(feat_name)

# C. MPM日数
features.append('DaysToNextMPM')

# ターゲット: BOJ_3の翌日の値 (ここでは絶対値を予測)
target_col = swap_cols[2] # BOJ_3
df_feats['Target'] = df_feats[target_col].shift(-1)

df_feats.dropna(inplace=True)
print(f'Total Features: {len(features)}')
print('\n--- Feature Samples ---')
display(df_feats[features].head())

In [ ]:
# 4. 学習と評価
X = df_feats[features]
y = df_feats['Target']

tscv = TimeSeriesSplit(n_splits=5)
all_mae, all_mse, all_dir = [], [], []

for tr, te in tscv.split(X):
    X_train, X_test = X.iloc[tr], X.iloc[te]
    y_train, y_test = y.iloc[tr], y.iloc[te]
    
    model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42, verbosity=-1)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='rmse', callbacks=[lgb.early_stopping(stopping_rounds=30)])
    
    y_pred = model.predict(X_test)
    all_mae.append(mean_absolute_error(y_test, y_pred))
    all_mse.append(mean_squared_error(y_test, y_pred))
    
    # 方向一致率 (BOJ_3の前日比予測精度)
    actual_change = y_test - df.loc[y_test.index, target_col].shift(0) # 修正: lag1ではなく現在の値と比較
    # 厳密には X_test 内の lag1 特徴量と比較すべき
    # ここでは単純に明日の値が今日の値より高いか低いかを評価
    ref_val = df_feats.loc[y_test.index, f'BOJ3_Spread_lag1'] + df_feats.loc[y_test.index, tona_col].shift(1) # 再構成
    # より正確な方向一致率計算:
    actual_dir = np.sign(y_test - df_feats.loc[y_test.index, target_col])
    pred_dir = np.sign(y_pred - df_feats.loc[y_test.index, target_col])
    all_dir.append(np.mean(actual_dir == pred_dir))

print(f'\n--- BOJ_3 Prediction with Custom Features ---')
print(f'MSE: {np.mean(all_mse):.7f}')
print(f'MAE: {np.mean(all_mae)*100:.3f} bps')
print(f'Directional Accuracy: {np.mean(all_dir):.2%}')

In [ ]:
# 5. Gain Importance
imp = pd.DataFrame({
    'feature': features, 
    'importance': model.booster_.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', data=imp)
plt.title('BOJ_3 Feature Importance (Gain)')
plt.show()

## 6. 特徴量の可視化
モデルに投入されている特徴量が、時間の経過とともにどのように変化しているかを確認します。

In [ ]:
# 6.1 主要スプレッドの推移 (水準の確認)
plt.figure(figsize=(15, 6))
for i in [1, 3, 8]:
    col = f'BOJ{i}_Spread_lag1'
    plt.plot(df_feats['日付'], df_feats[col], label=col)
plt.title('Comparison of BOJ-TONA Spreads (Lag1)')
plt.ylabel('Spread (%)')
plt.legend()
plt.show()

# 6.2 MA乖離の視覚化 (シグナルの確認)
# BOJ3スプレッドとその5日MA乖離をプロット
fig, ax1 = plt.subplots(figsize=(15, 6))
ax1.plot(df_feats['日付'], df_feats['BOJ3_Spread_lag1'], color='blue', alpha=0.3, label='BOJ3_Spread_lag1')
ax1.set_ylabel('Spread Level (%)', color='blue')

ax2 = ax1.twinx()
ax2.bar(df_feats['日付'], df_feats['BOJ3_Spread_diff_MA5'], color='red', alpha=0.5, label='Diff_MA5')
ax2.set_ylabel('MA5 Difference (%)', color='red')
ax2.axhline(0, color='black', lw=1)

plt.title('BOJ3 Spread Level vs its 5-day MA Momentum')
plt.show()

# 6.3 相関ヒートマップ
# 重要度上位10個とTargetの相関を確認
top_features = imp['feature'].tolist()[:10]
corr_matrix = df_feats[top_features + ['Target']].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap: Top Features & Target')
plt.show()